# RF-DETR Seg Large Optimization v1 (16GB VRAM, RTX 5060 Ti)

Transformer-focused training + evaluation pipeline, upgraded from the 4GB `RFDETRSegSmall`
iteration to a larger / more robust segmentation backbone now that we run on a 16GB GPU.

**Runs in this notebook**
- `seg_large_r504_auto` — ablation (input 504 px)
- `seg_large_r672_auto` — **champion** (input 672 px; conf* = 0.44; used in the paper)

Baseline RF-DETR Seg Small (4GB) is archived under `archive/experiments/v2_rfdetr_seg_small/`.

This notebook keeps the same artifact contract as the archived Small iteration so downstream
selectors and showcase notebooks stay compatible:

- dataset preprocessing + integrity validation (COCO, segmentation)
- ground-truth mask sanity overlays (preprocessing visualization)
- constrained training sweep on the larger model
- per-run confidence calibration on the validation set
- per-instance and global extraction reports
- IoU visual overlays
- inference-time studies (latency, throughput, peak VRAM)

Class of interest: `microcotiledone` (single foreground class).

In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import rfdetr
from rfdetr import (
    RFDETRSegNano,
    RFDETRSegSmall,
    RFDETRSegMedium,
    RFDETRSegLarge,
    RFDETRSegXLarge,
    RFDETRSeg2XLarge,
)

c:\Users\edual\miniconda3\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\edual\miniconda3\envs\pytorch\Lib\site-packages\rfdetr\models\weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, args_mapping={"train_config": None}, deprecated_in="1.7.0", remove_in="1.9.0", num_warns=-1)


In [ ]:
def discover_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists() and (p / 'v2_rfdetr_seg_large_opt_v1').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root')


REPO_ROOT = discover_repo_root()
PROJECT_ROOT = REPO_ROOT / 'v2_rfdetr_seg_large_opt_v1'
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
RUNS_ROOT = ARTIFACTS_ROOT / 'runs'
REPORTS_ROOT = ARTIFACTS_ROOT / 'reports'
VIZ_ROOT = ARTIFACTS_ROOT / 'iou_viz'
BENCH_ROOT = ARTIFACTS_ROOT / 'benchmarks'
PREPROC_ROOT = ARTIFACTS_ROOT / 'preprocessing'
for p in [RUNS_ROOT, REPORTS_ROOT, VIZ_ROOT, BENCH_ROOT, PREPROC_ROOT]:
    p.mkdir(parents=True, exist_ok=True)


def resolve_dataset_dir() -> Path:
    # Dataset lives outside the repo. Support several machine layouts + env override
    # so the notebook is portable between the workstation and this SSH box.
    env = os.environ.get('PLACENTA_COCO_DIR')
    candidates = []
    if env:
        candidates.append(Path(env))
    candidates += [
        REPO_ROOT.parent / 'datasets' / 'dataset_v2_coco_rf_detr',
        REPO_ROOT.parent.parent / 'dataset_v2_coco_rf_detr',
        Path(r'C:/Users/edual/projeto-placentas/datasets/dataset_v2_coco_rf_detr'),
    ]
    for c in candidates:
        if (c / 'train' / '_annotations.coco.json').exists():
            return c.resolve()
    raise FileNotFoundError(
        'Could not locate dataset_v2_coco_rf_detr. Set PLACENTA_COCO_DIR or check paths.\n'
        + '\n'.join(f'  tried: {c}' for c in candidates)
    )


DATASET_DIR = resolve_dataset_dir()

# ---- Evaluation constants (kept identical to prior iterations for comparability) ----
IMG_W = 640
IMG_H = 640
IOU_THRESHOLD = 0.5
# Corrected 2026-09-07: anisotropic 4140x3096 -> 640x640 stretch means the 50um/72px
# horizontal scale bar measurement does not apply uniformly to Y; multiply by
# H_native/W_native = 3096/4140 = 0.747826 (cross-checked via direct native-scale-bar measurement).
AREA_FACTOR = (50 / 72) ** 2 * (3096 / 4140)  # px^2 -> um^2 calibration used across the project
CONF_CANDIDATES = [round(x, 2) for x in np.arange(0.20, 0.71, 0.02)]

# ---- Model selection --------------------------------------------------------------
# Bigger / more robust than the 4GB RFDETRSegSmall baseline. Swap here to scale up
# (RFDETRSegXLarge / RFDETRSeg2XLarge) if VRAM headroom allows.
SEG_MODEL_REGISTRY = {
    'RFDETRSegNano': RFDETRSegNano,
    'RFDETRSegSmall': RFDETRSegSmall,
    'RFDETRSegMedium': RFDETRSegMedium,
    'RFDETRSegLarge': RFDETRSegLarge,
    'RFDETRSegXLarge': RFDETRSegXLarge,
    'RFDETRSeg2XLarge': RFDETRSeg2XLarge,
}
SEG_MODEL_NAME = 'RFDETRSegLarge'
SEG_MODEL_CLASS = SEG_MODEL_REGISTRY[SEG_MODEL_NAME]
# Seg variants require resolution divisible by 24 (Nano: 12).
RES_DIVISOR = 12 if SEG_MODEL_NAME == 'RFDETRSegNano' else 24

# Toggle to False to skip training and only (re)evaluate existing run folders.
RUN_TRAINING = True

# 16GB-oriented sweep. batch_size='auto' lets RF-DETR probe the GPU and pick the
# largest safe per-step batch, then use grad_accum to reach auto_batch_target_effective
# (the effective batch that actually drives optimization). This exploits the more
# powerful card without hand-tuning per-resolution batch sizes or risking OOM.
# To pin values manually instead, set an int batch_size + grad_accum_steps.
RUN_MATRIX = [
    {
        'run_name': 'seg_large_r504_auto',
        'epochs': 100,
        'batch_size': 'auto',
        'auto_batch_target_effective': 16,
        'lr': 1e-4,
        'resolution': 504,
        'gradient_checkpointing': True,
        'early_stopping_patience': 20,
    },
    {
        'run_name': 'seg_large_r672_auto',
        'epochs': 100,
        'batch_size': 'auto',
        'auto_batch_target_effective': 16,
        'lr': 1e-4,
        'resolution': 672,
        'gradient_checkpointing': True,
        'early_stopping_patience': 20,
    },
]

for cfg in RUN_MATRIX:
    assert cfg['resolution'] % RES_DIVISOR == 0, (
        f"resolution {cfg['resolution']} must be divisible by {RES_DIVISOR} for {SEG_MODEL_NAME}"
    )

print('rfdetr:      ', getattr(rfdetr, '__version__', '?'))
print('REPO_ROOT:   ', REPO_ROOT)
print('DATASET_DIR: ', DATASET_DIR)
print('ARTIFACTS:   ', ARTIFACTS_ROOT)
print('MODEL:       ', SEG_MODEL_NAME)
print('RUN_TRAINING:', RUN_TRAINING)
print('RUNS:        ', [r['run_name'] for r in RUN_MATRIX])
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:         ', props.name, f'| {props.total_memory / 1024**3:.1f} GB',
          '| bf16', torch.cuda.is_bf16_supported())
else:
    print('GPU:          NOT AVAILABLE (training will be very slow on CPU)')

rfdetr:       ?
REPO_ROOT:    C:\Users\edual\projeto-placentas\codespace
DATASET_DIR:  C:\Users\edual\projeto-placentas\datasets\dataset_v2_coco_rf_detr
ARTIFACTS:    C:\Users\edual\projeto-placentas\codespace\v2_rfdetr_seg_large_opt_v1\artifacts
MODEL:        RFDETRSegLarge
RUN_TRAINING: True
RUNS:         ['seg_large_r504_auto', 'seg_large_r672_auto']
GPU:          NVIDIA GeForce RTX 5060 Ti | 15.9 GB | bf16 True


## 1. Dataset preprocessing & integrity validation

RF-DETR consumes the Roboflow COCO layout directly and applies its own resize +
ImageNet normalization internally (the normalization is tied to the DINOv2 pretrained
backbone, so we deliberately do **not** override it). What we validate here before
committing GPU hours:

- every split has a readable `_annotations.coco.json`
- every referenced image file exists and opens, with the expected 640x640 size
- category mapping is what we expect (`microcotiledone` foreground)
- per-image instance-count distribution (microcotiledones are many & small)

In [3]:
def load_coco(split: str) -> dict | None:
    ann_path = DATASET_DIR / split / '_annotations.coco.json'
    if not ann_path.exists():
        return None
    return json.loads(ann_path.read_text(encoding='utf-8'))


def validate_split(split: str) -> Dict:
    data = load_coco(split)
    if data is None:
        print(f'[{split}] missing (skipped)')
        return {}

    images = data.get('images', [])
    anns = data.get('annotations', [])
    cats = {int(c['id']): c.get('name', '') for c in data.get('categories', [])}

    img_dir = DATASET_DIR / split
    missing_files, bad_size, unreadable = [], [], []
    for im in images:
        fp = img_dir / im['file_name']
        if not fp.exists():
            missing_files.append(im['file_name'])
            continue
        img = cv2.imread(str(fp))
        if img is None:
            unreadable.append(im['file_name'])
            continue
        h, w = img.shape[:2]
        if (w, h) != (int(im['width']), int(im['height'])):
            bad_size.append((im['file_name'], (w, h), (im['width'], im['height'])))

    seg_n = sum(1 for a in anns if a.get('segmentation'))
    per_img: Dict[int, int] = {}
    for a in anns:
        per_img[int(a['image_id'])] = per_img.get(int(a['image_id']), 0) + 1
    counts = list(per_img.values()) or [0]

    print(f'[{split}] images={len(images)} annotations={len(anns)} with_seg={seg_n}')
    print(f'[{split}] categories={cats}')
    print(f'[{split}] instances/image: min={min(counts)} max={max(counts)} mean={np.mean(counts):.2f}')
    print(f'[{split}] missing_files={len(missing_files)} unreadable={len(unreadable)} bad_size={len(bad_size)}')
    if missing_files:
        print('   e.g. missing:', missing_files[:3])
    if bad_size:
        print('   e.g. bad_size:', bad_size[:3])

    return {
        'split': split,
        'images': len(images),
        'annotations': len(anns),
        'with_segmentation': seg_n,
        'categories': cats,
        'inst_min': int(min(counts)),
        'inst_max': int(max(counts)),
        'inst_mean': float(np.mean(counts)),
        'missing_files': missing_files,
        'unreadable': unreadable,
        'bad_size': [f for f, _, _ in bad_size],
    }


preproc_summary = {s: validate_split(s) for s in ('train', 'valid', 'test')}
preproc_summary = {k: v for k, v in preproc_summary.items() if v}

for split, rep in preproc_summary.items():
    assert not rep['missing_files'], f'{split}: missing image files: {rep["missing_files"][:5]}'
    assert not rep['unreadable'], f'{split}: unreadable images: {rep["unreadable"][:5]}'

(PREPROC_ROOT / 'dataset_validation.json').write_text(
    json.dumps(preproc_summary, indent=2), encoding='utf-8'
)
print('\nSaved:', PREPROC_ROOT / 'dataset_validation.json')

[train] images=153 annotations=4753 with_seg=4752
[train] categories={0: 'objects', 1: 'microcotiledone'}
[train] instances/image: min=10 max=70 mean=31.07
[train] missing_files=0 unreadable=0 bad_size=0
[valid] images=27 annotations=852 with_seg=852
[valid] categories={0: 'objects', 1: 'microcotiledone'}
[valid] instances/image: min=13 max=48 mean=31.56
[valid] missing_files=0 unreadable=0 bad_size=0
[test] missing (skipped)

Saved: C:\Users\edual\projeto-placentas\codespace\v2_rfdetr_seg_large_opt_v1\artifacts\preprocessing\dataset_validation.json


In [4]:
def poly_ann_to_mask(ann: Dict, height: int, width: int) -> np.ndarray:
    seg = ann.get('segmentation', None)
    mask = np.zeros((height, width), dtype=np.uint8)
    if not seg:
        return mask
    if isinstance(seg, list):
        for poly in seg:
            pts = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
            cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
        return mask
    if isinstance(seg, dict):
        try:
            from pycocotools import mask as mask_utils
            return mask_utils.decode(seg).astype(np.uint8)
        except Exception:
            return mask
    return mask


def preview_ground_truth(split: str = 'train', n: int = 6) -> Path:
    data = load_coco(split)
    images = data.get('images', [])[:n]
    anns_by_img: Dict[int, List[Dict]] = {}
    for a in data.get('annotations', []):
        anns_by_img.setdefault(int(a['image_id']), []).append(a)

    out_dir = PREPROC_ROOT / 'gt_preview'
    out_dir.mkdir(parents=True, exist_ok=True)
    img_dir = DATASET_DIR / split

    for im in images:
        h, w = int(im['height']), int(im['width'])
        img = cv2.cvtColor(cv2.imread(str(img_dir / im['file_name'])), cv2.COLOR_BGR2RGB)
        combined = np.zeros((h, w), dtype=np.uint8)
        for a in anns_by_img.get(int(im['id']), []):
            combined = np.logical_or(combined, poly_ann_to_mask(a, h, w)).astype(np.uint8)
        overlay = img.copy()
        overlay[combined == 1] = (overlay[combined == 1] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        n_inst = len(anns_by_img.get(int(im['id']), []))
        fig.suptitle(f"{im['file_name']} | GT instances={n_inst}")
        axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
        axes[1].imshow(overlay); axes[1].set_title('GT masks (green)'); axes[1].axis('off')
        plt.savefig(out_dir / f"gt_{Path(im['file_name']).stem}.png", dpi=130, bbox_inches='tight')
        plt.close(fig)
    return out_dir


gt_preview_dir = preview_ground_truth('train', n=6)
print('GT preview saved to:', gt_preview_dir)

GT preview saved to: C:\Users\edual\projeto-placentas\codespace\v2_rfdetr_seg_large_opt_v1\artifacts\preprocessing\gt_preview


## 2. Training sweep (larger model)

`gradient_checkpointing=True` trades a little compute for a large memory saving, which
lets us keep a real batch size and a higher input resolution than the 4GB runs.
Mixed precision (`amp_dtype='auto'` -> bf16 on this GPU) is handled internally by RF-DETR.
`tensorboard=False` because the logger package is not installed in this environment.

In [5]:
def run_training(cfg: Dict) -> Path:
    output_dir = RUNS_ROOT / cfg['run_name']
    output_dir.mkdir(parents=True, exist_ok=True)

    train_kwargs = dict(
        dataset_dir=str(DATASET_DIR),
        output_dir=str(output_dir),
        epochs=int(cfg['epochs']),
        lr=float(cfg['lr']),
        gradient_checkpointing=bool(cfg['gradient_checkpointing']),
        resolution=int(cfg['resolution']),
        early_stopping=True,
        early_stopping_patience=int(cfg['early_stopping_patience']),
        tensorboard=False,
        progress_bar='tqdm',
        seed=0,
    )

    if cfg['batch_size'] == 'auto':
        # RF-DETR probes the GPU for the largest safe per-step batch, then uses
        # grad_accum internally to reach this effective batch. Don't pass a fixed
        # grad_accum_steps in this mode.
        train_kwargs['batch_size'] = 'auto'
        train_kwargs['auto_batch_target_effective'] = int(cfg.get('auto_batch_target_effective', 16))
    else:
        train_kwargs['batch_size'] = int(cfg['batch_size'])
        train_kwargs['grad_accum_steps'] = int(cfg['grad_accum_steps'])

    model = SEG_MODEL_CLASS()
    model.train(**train_kwargs)
    return output_dir


trained_run_dirs: Dict[str, Path] = {}
if RUN_TRAINING:
    for cfg in RUN_MATRIX:
        print(f"\n=== Training {cfg['run_name']} ({SEG_MODEL_NAME}) ===")
        out = run_training(cfg)
        trained_run_dirs[cfg['run_name']] = out
        print('Finished:', out)
else:
    for cfg in RUN_MATRIX:
        trained_run_dirs[cfg['run_name']] = RUNS_ROOT / cfg['run_name']

trained_run_dirs

Val (Epoch 29/100) — Overall Metrics                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃     segm mAP    ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │ 50:95  │   50   │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.8152 │ 0.9261 │ 0.8567 │ 0.8899 │ 0.9119 │ 0.9071 │ 0.9167 │ 0.7932 │ 0.9415 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┴────────┘
               Val (Epoch 29/100) — Per-class Metrics                
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class           ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ microcotiledone │   0.8152 │ 0.8899 │ 0.9119 │    0.9071 │ 0.9167 │
└─────────────────┴──────────┴────────┴────────┴───────────┴────────┘

Epoch 28: 100%|██████████| 160/160 [00:59<00:00,  2.67it/s, loss=6.600, loss_cls=0.383, loss_box=0.0133, loss_giou=0.0876, mask_ce=0.00773, mask_dice=0.0369, val/loss=7.500, val/mAP_50_95=0.815, val/mAP_50=0.926, val/ema_mAP_50_95=0.821, val/F1=0.912]

Monitored metric __rfdetr_effective_map__ did not improve in the last 20 records. Best score: 0.825. Signaling Trainer to stop.


Epoch 28: 100%|██████████| 160/160 [01:01<00:00,  2.58it/s, loss=6.600, loss_cls=0.383, loss_box=0.0133, loss_giou=0.0876, mask_ce=0.00773, mask_dice=0.0369, val/loss=7.500, val/mAP_50_95=0.815, val/mAP_50=0.926, val/ema_mAP_50_95=0.821, val/F1=0.912]
[2026-07-14 19:11:03] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.8248, ema=0.8249)
Finished: C:\Users\edual\projeto-placentas\codespace\v2_rfdetr_seg_large_opt_v1\artifacts\runs\seg_large_r672_auto


{'seg_large_r504_auto': WindowsPath('C:/Users/edual/projeto-placentas/codespace/v2_rfdetr_seg_large_opt_v1/artifacts/runs/seg_large_r504_auto'),
 'seg_large_r672_auto': WindowsPath('C:/Users/edual/projeto-placentas/codespace/v2_rfdetr_seg_large_opt_v1/artifacts/runs/seg_large_r672_auto')}

In [6]:
def pick_checkpoint(run_dir: Path) -> Path:
    candidates = [
        run_dir / 'checkpoint_best_total.pth',
        run_dir / 'checkpoint_best_regular.pth',
        run_dir / 'checkpoint_best_ema.pth',
    ]
    for c in candidates:
        if c.is_file():
            return c
    raise FileNotFoundError(f'No best checkpoint found in {run_dir}')


def load_coco_valid():
    ann_path = DATASET_DIR / 'valid' / '_annotations.coco.json'
    coco = json.loads(ann_path.read_text(encoding='utf-8'))
    imgs = coco.get('images', [])
    anns = coco.get('annotations', [])

    imgs_by_id = {int(i['id']): i for i in imgs}
    anns_by_img: Dict[int, List[Dict]] = {}
    for a in anns:
        anns_by_img.setdefault(int(a['image_id']), []).append(a)
    return imgs, imgs_by_id, anns_by_img


VAL_IMGS, VAL_IMGS_BY_ID, VAL_ANNS_BY_IMG = load_coco_valid()
VALID_IMG_DIR = DATASET_DIR / 'valid'
print('valid images:', len(VAL_IMGS))

valid images: 27


In [7]:
def ann_to_mask(ann: Dict, height: int, width: int) -> np.ndarray:
    return poly_ann_to_mask(ann, height, width)


def get_gt_masks_for_image(img_id: int) -> Tuple[List[np.ndarray], int, int]:
    info = VAL_IMGS_BY_ID[int(img_id)]
    h, w = int(info['height']), int(info['width'])
    anns = VAL_ANNS_BY_IMG.get(int(img_id), [])
    masks = [ann_to_mask(a, h, w) for a in anns]
    return masks, h, w


def get_pred_masks_and_conf(det) -> Tuple[List[np.ndarray], List[float]]:
    masks: List[np.ndarray] = []
    confs: List[float] = []

    pred_mask = getattr(det, 'mask', None)
    pred_conf = getattr(det, 'confidence', None)

    if pred_mask is not None:
        arr = np.asarray(pred_mask)
        if arr.ndim == 2:
            masks.append((arr > 0).astype(np.uint8))
        elif arr.ndim == 3:
            for m in arr:
                masks.append((m > 0).astype(np.uint8))

    if pred_conf is not None:
        c = np.asarray(pred_conf).reshape(-1)
        confs = [float(x) for x in c]

    if len(confs) != len(masks):
        confs = [1.0] * len(masks)

    return masks, confs


def iou_binary(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0


def greedy_match(pred_masks: List[np.ndarray], gt_masks: List[np.ndarray], iou_thr: float = 0.5):
    candidates: List[Tuple[float, int, int]] = []
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            score = iou_binary(pm, gm)
            if score >= iou_thr:
                candidates.append((score, i, j))
    candidates.sort(key=lambda x: x[0], reverse=True)

    used_p, used_g = set(), set()
    pairs = []
    for score, i, j in candidates:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        pairs.append((i, j, score))
    return pairs, used_p, used_g

In [8]:
def eval_run_at_conf(infer_model, conf_thr: float) -> Dict[str, float]:
    tp = fp = fn = 0
    ious = []
    gt_total_area = 0
    pred_total_area = 0

    for info in VAL_IMGS:
        img_id = int(info['id'])
        img_name = info['file_name']
        img_path = VALID_IMG_DIR / img_name

        gt_masks, h, w = get_gt_masks_for_image(img_id)
        gt_total_area += sum(int(gm.sum()) for gm in gt_masks)

        det = infer_model.predict(str(img_path), threshold=float(conf_thr))
        pred_masks, pred_confs = get_pred_masks_and_conf(det)
        pred_masks = [m for m, c in zip(pred_masks, pred_confs) if c >= conf_thr]

        resized_preds = [cv2.resize(pm.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) for pm in pred_masks]
        pred_masks = [(pm > 0).astype(np.uint8) for pm in resized_preds]
        pred_total_area += sum(int(pm.sum()) for pm in pred_masks)

        pairs, used_p, used_g = greedy_match(pred_masks, gt_masks, iou_thr=IOU_THRESHOLD)
        tp += len(pairs)
        fp += len(pred_masks) - len(used_p)
        fn += len(gt_masks) - len(used_g)
        ious.extend([p[2] for p in pairs])

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    mean_iou = float(np.mean(ious)) if ious else 0.0
    area_rel_error = abs(pred_total_area - gt_total_area) / max(gt_total_area, 1)
    score = (0.6 * f1) + (0.3 * mean_iou) + (0.1 * (1.0 - area_rel_error))

    return {
        'conf': float(conf_thr),
        'tp': int(tp),
        'fp': int(fp),
        'fn': int(fn),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'mean_iou': float(mean_iou),
        'gt_total_area_px': int(gt_total_area),
        'pred_total_area_px': int(pred_total_area),
        'area_rel_error': float(area_rel_error),
        'score': float(score),
    }

In [9]:
run_summaries = []
all_conf_rows = []

for cfg in RUN_MATRIX:
    run_name = cfg['run_name']
    run_dir = trained_run_dirs[run_name]
    ckpt = pick_checkpoint(run_dir)

    print(f'\n=== Confidence sweep: {run_name} ===')
    infer_model = SEG_MODEL_CLASS(pretrain_weights=str(ckpt))

    rows = []
    for conf_thr in CONF_CANDIDATES:
        row = eval_run_at_conf(infer_model, conf_thr=conf_thr)
        row['run_name'] = run_name
        rows.append(row)
        all_conf_rows.append(row)

    sweep_df = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
    sweep_csv = BENCH_ROOT / f'validation_conf_sweep_{run_name}.csv'
    sweep_df.to_csv(sweep_csv, index=False)

    best = sweep_df.iloc[0].to_dict()
    best['checkpoint'] = str(ckpt)
    best['sweep_csv'] = str(sweep_csv)
    run_summaries.append(best)

    print('best conf:', best['conf'], 'score:', round(float(best['score']), 4))

runs_df = pd.DataFrame(run_summaries).sort_values('score', ascending=False).reset_index(drop=True)
runs_df.to_csv(BENCH_ROOT / 'run_ranking.csv', index=False)
runs_df.head(len(runs_df))

[2026-07-14 19:11:04] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-14 19:11:04] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.



=== Confidence sweep: seg_large_r504_auto ===


[2026-07-14 19:11:04] [WARNING] rf-detr - Checkpoint has 2 classes but model is configured for 90. Using checkpoint class count (2). Pass num_classes=2 to suppress this warning.
[2026-07-14 19:11:04] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=200 from tensor rows 2600 ÷ ckpt_group_detr=13.
[2026-07-14 19:11:04] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.optimize_for_inference(dtype=torch.float16).
[2026-07-14 19:20:18] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-14 19:20:18] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


best conf: 0.4 score: 0.9074

=== Confidence sweep: seg_large_r672_auto ===


[2026-07-14 19:20:18] [WARNING] rf-detr - Checkpoint has 2 classes but model is configured for 90. Using checkpoint class count (2). Pass num_classes=2 to suppress this warning.
[2026-07-14 19:20:18] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=200 from tensor rows 2600 ÷ ckpt_group_detr=13.
[2026-07-14 19:20:18] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.optimize_for_inference(dtype=torch.float16).


best conf: 0.44 score: 0.9116


,conf,tp,fp,fn,precision,recall,f1,mean_iou,gt_total_area_px,pred_total_area_px,area_rel_error,score,run_name,checkpoint,sweep_csv
0,0.44,775,93,77,0.892857,0.909624,0.901163,0.905417,4109860,4081240,0.006964,0.911626,seg_large_r672_auto,C:\Users\edual\projeto-placentas\codespace\v2_...,C:\Users\edual\projeto-placentas\codespace\v2_...
1,0.40,785,103,67,0.884009,0.921362,0.902299,0.893446,4109860,4026936,0.020177,0.907395,seg_large_r504_auto,C:\Users\edual\projeto-placentas\codespace\v2_...,C:\Users\edual\projeto-placentas\codespace\v2_...


In [10]:
if len(runs_df) == 0:
    raise RuntimeError('No run summaries found.')

CHAMP = runs_df.iloc[0].to_dict()
CHAMP_RUN = CHAMP['run_name']
CHAMP_CONF = float(CHAMP['conf'])
CHAMP_CKPT = Path(CHAMP['checkpoint'])

print('Champion run: ', CHAMP_RUN)
print('Champion conf:', CHAMP_CONF)
print('Checkpoint:   ', CHAMP_CKPT)

(BENCH_ROOT / 'champion_run.json').write_text(
    json.dumps(
        {
            'model': SEG_MODEL_NAME,
            'run_name': CHAMP_RUN,
            'best_conf': CHAMP_CONF,
            'checkpoint': str(CHAMP_CKPT),
            'score': float(CHAMP['score']),
        },
        indent=2,
    ),
    encoding='utf-8',
)

Champion run:  seg_large_r672_auto
Champion conf: 0.44
Checkpoint:    C:\Users\edual\projeto-placentas\codespace\v2_rfdetr_seg_large_opt_v1\artifacts\runs\seg_large_r672_auto\checkpoint_best_total.pth


279

In [11]:
def evaluate_and_export_reports(run_name: str, ckpt_path: Path, conf_thr: float):
    infer_model = SEG_MODEL_CLASS(pretrain_weights=str(ckpt_path))

    run_report_dir = REPORTS_ROOT / run_name
    run_viz_dir = VIZ_ROOT / run_name
    run_report_dir.mkdir(parents=True, exist_ok=True)
    run_viz_dir.mkdir(parents=True, exist_ok=True)

    instance_rows = []
    totals_rows = []

    for info in VAL_IMGS:
        img_id = int(info['id'])
        img_name = info['file_name']
        img_path = VALID_IMG_DIR / img_name

        gt_masks, h, w = get_gt_masks_for_image(img_id)
        det = infer_model.predict(str(img_path), threshold=float(conf_thr))
        pred_masks, pred_confs = get_pred_masks_and_conf(det)
        pred_masks = [m for m, c in zip(pred_masks, pred_confs) if c >= conf_thr]
        pred_masks = [(cv2.resize(pm.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8) for pm in pred_masks]

        pairs, used_p, used_g = greedy_match(pred_masks, gt_masks, iou_thr=IOU_THRESHOLD)

        gt_area_total = int(sum(int(gm.sum()) for gm in gt_masks))
        pred_area_total = int(sum(int(pm.sum()) for pm in pred_masks))

        totals_rows.append(
            {
                'Image': img_name,
                'GT_Count': len(gt_masks),
                'AI_Count': len(pred_masks),
                'Matched_Count': len(pairs),
                'FP_Count': len(pred_masks) - len(pairs),
                'FN_Count': len(gt_masks) - len(pairs),
                'GT_Area_px': gt_area_total,
                'AI_Area_px': pred_area_total,
                'GT_Area_um2': round(gt_area_total * AREA_FACTOR, 4),
                'AI_Area_um2': round(pred_area_total * AREA_FACTOR, 4),
                'Conf': conf_thr,
                'Run': run_name,
            }
        )

        for p_idx, g_idx, score in pairs:
            p_area = int(pred_masks[p_idx].sum())
            g_area = int(gt_masks[g_idx].sum())
            instance_rows.append(
                {
                    'Image': img_name,
                    'Match_Type': 'TP',
                    'AI_Index': p_idx,
                    'GT_Index': g_idx,
                    'IoU': round(score, 6),
                    'GT_Area_px': g_area,
                    'AI_Area_px': p_area,
                    'GT_Area_um2': round(g_area * AREA_FACTOR, 4),
                    'AI_Area_um2': round(p_area * AREA_FACTOR, 4),
                    'Conf': conf_thr,
                    'Run': run_name,
                }
            )

        for p_idx, pm in enumerate(pred_masks):
            if p_idx in used_p:
                continue
            p_area = int(pm.sum())
            instance_rows.append(
                {
                    'Image': img_name,
                    'Match_Type': 'FP',
                    'AI_Index': p_idx,
                    'GT_Index': -1,
                    'IoU': 0.0,
                    'GT_Area_px': 0,
                    'AI_Area_px': p_area,
                    'GT_Area_um2': 0.0,
                    'AI_Area_um2': round(p_area * AREA_FACTOR, 4),
                    'Conf': conf_thr,
                    'Run': run_name,
                }
            )

        for g_idx, gm in enumerate(gt_masks):
            if g_idx in used_g:
                continue
            g_area = int(gm.sum())
            instance_rows.append(
                {
                    'Image': img_name,
                    'Match_Type': 'FN',
                    'AI_Index': -1,
                    'GT_Index': g_idx,
                    'IoU': 0.0,
                    'GT_Area_px': g_area,
                    'AI_Area_px': 0,
                    'GT_Area_um2': round(g_area * AREA_FACTOR, 4),
                    'AI_Area_um2': 0.0,
                    'Conf': conf_thr,
                    'Run': run_name,
                }
            )

        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        gt_combined = np.zeros((h, w), dtype=np.uint8)
        pred_combined = np.zeros((h, w), dtype=np.uint8)
        for gm in gt_masks:
            gt_combined = np.logical_or(gt_combined, gm).astype(np.uint8)
        for pm in pred_masks:
            pred_combined = np.logical_or(pred_combined, pm).astype(np.uint8)

        inter = np.logical_and(gt_combined, pred_combined).sum()
        union = np.logical_or(gt_combined, pred_combined).sum()
        img_iou = (inter / union) if union > 0 else 0.0

        overlay = img.copy()
        overlay[gt_combined == 1] = (overlay[gt_combined == 1] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
        overlay[pred_combined == 1] = (overlay[pred_combined == 1] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
        overlap = np.logical_and(gt_combined, pred_combined)
        overlay[overlap] = (overlay[overlap] * 0.5 + np.array([255, 255, 0]) * 0.5).astype(np.uint8)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(f"{img_name} | IoU={img_iou:.3f} | GT={len(gt_masks)} | Pred={len(pred_masks)}")
        axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
        axes[1].imshow(overlay); axes[1].set_title('GT=green AI=red overlap=yellow'); axes[1].axis('off')
        plt.savefig(run_viz_dir / f"iou_viz_{Path(img_name).stem}.png", dpi=140, bbox_inches='tight')
        plt.close(fig)

    instance_csv = run_report_dir / 'placenta_instance_report_rfdetr.csv'
    totals_csv = run_report_dir / 'placenta_totals_report_rfdetr.csv'
    pd.DataFrame(instance_rows).to_csv(instance_csv, index=False)
    pd.DataFrame(totals_rows).to_csv(totals_csv, index=False)

    return instance_csv, totals_csv, run_viz_dir


inst_csv, totals_csv, viz_dir = evaluate_and_export_reports(CHAMP_RUN, CHAMP_CKPT, CHAMP_CONF)
print('Instance report:', inst_csv)
print('Totals report:  ', totals_csv)
print('Viz dir:        ', viz_dir)

[2026-07-14 19:29:39] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-14 19:29:39] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-14 19:29:39] [WARNING] rf-detr - Checkpoint has 2 classes but model is configured for 90. Using checkpoint class count (2). Pass num_classes=2 to suppress this warning.
[2026-07-14 19:29:39] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=200 from tensor rows 2600 ÷ ckpt_group_detr=13.
[2026-07-14 19:29:39] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.optimize_for_inference(dtype=torch.float16).


Instance report: C:\Users\edual\projeto-placentas\codespace\v2_rfdetr_seg_large_opt_v1\artifacts\reports\seg_large_r672_auto\placenta_instance_report_rfdetr.csv
Totals report:   C:\Users\edual\projeto-placentas\codespace\v2_rfdetr_seg_large_opt_v1\artifacts\reports\seg_large_r672_auto\placenta_totals_report_rfdetr.csv
Viz dir:         C:\Users\edual\projeto-placentas\codespace\v2_rfdetr_seg_large_opt_v1\artifacts\iou_viz\seg_large_r672_auto


In [12]:
def benchmark_inference(run_name: str, ckpt_path: Path, conf_thr: float, repeats: int = 3, warmup: int = 1) -> Path:
    infer_model = SEG_MODEL_CLASS(pretrain_weights=str(ckpt_path))

    if hasattr(infer_model, 'optimize_for_inference'):
        try:
            infer_model.optimize_for_inference()
        except Exception as e:
            print(f'optimize_for_inference skipped: {e}')

    img_paths = [VALID_IMG_DIR / i['file_name'] for i in VAL_IMGS]

    for _ in range(warmup):
        for p in img_paths:
            _ = infer_model.predict(str(p), threshold=float(conf_thr))

    timings = []
    peak_mem_mb = None

    for _ in range(repeats):
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        t0 = time.time()
        for p in img_paths:
            _ = infer_model.predict(str(p), threshold=float(conf_thr))

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

        timings.append(time.time() - t0)

    mean_elapsed = float(np.mean(timings))
    std_elapsed = float(np.std(timings))
    ips = (len(img_paths) / mean_elapsed) if mean_elapsed > 0 else 0.0

    payload = {
        'model': SEG_MODEL_NAME,
        'run_name': run_name,
        'checkpoint': str(ckpt_path),
        'conf': float(conf_thr),
        'n_images': len(img_paths),
        'repeats': repeats,
        'warmup': warmup,
        'elapsed_sec_mean': mean_elapsed,
        'elapsed_sec_std': std_elapsed,
        'images_per_sec': ips,
        'peak_gpu_mem_mb': peak_mem_mb,
    }

    out_json = BENCH_ROOT / f'inference_benchmark_{run_name}.json'
    out_json.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    return out_json


bench_json = benchmark_inference(CHAMP_RUN, CHAMP_CKPT, CHAMP_CONF, repeats=3, warmup=1)
print('Benchmark:', bench_json)

[2026-07-14 19:30:06] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-14 19:30:06] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-14 19:30:06] [WARNING] rf-detr - Checkpoint has 2 classes but model is configured for 90. Using checkpoint class count (2). Pass num_classes=2 to suppress this warning.
[2026-07-14 19:30:06] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=200 from tensor rows 2600 ÷ ckpt_group_detr=13.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Benchmark: C:\Users\edual\projeto-placentas\codespace\v2_rfdetr_seg_large_opt_v1\artifacts\benchmarks\inference_benchmark_seg_large_r672_auto.json


In [13]:
final_summary = {
    'model': SEG_MODEL_NAME,
    'champion_run': CHAMP_RUN,
    'champion_conf': CHAMP_CONF,
    'champion_checkpoint': str(CHAMP_CKPT),
    'dataset_dir': str(DATASET_DIR),
    'run_ranking_csv': str(BENCH_ROOT / 'run_ranking.csv'),
    'dataset_validation_json': str(PREPROC_ROOT / 'dataset_validation.json'),
    'instance_report_csv': str(inst_csv),
    'totals_report_csv': str(totals_csv),
    'iou_viz_dir': str(viz_dir),
    'inference_benchmark_json': str(bench_json),
}

final_summary_path = BENCH_ROOT / 'final_summary.json'
final_summary_path.write_text(json.dumps(final_summary, indent=2), encoding='utf-8')
print(json.dumps(final_summary, indent=2))
print('Saved:', final_summary_path)

{
  "model": "RFDETRSegLarge",
  "champion_run": "seg_large_r672_auto",
  "champion_conf": 0.44,
  "champion_checkpoint": "C:\\Users\\edual\\projeto-placentas\\codespace\\v2_rfdetr_seg_large_opt_v1\\artifacts\\runs\\seg_large_r672_auto\\checkpoint_best_total.pth",
  "dataset_dir": "C:\\Users\\edual\\projeto-placentas\\datasets\\dataset_v2_coco_rf_detr",
  "run_ranking_csv": "C:\\Users\\edual\\projeto-placentas\\codespace\\v2_rfdetr_seg_large_opt_v1\\artifacts\\benchmarks\\run_ranking.csv",
  "dataset_validation_json": "C:\\Users\\edual\\projeto-placentas\\codespace\\v2_rfdetr_seg_large_opt_v1\\artifacts\\preprocessing\\dataset_validation.json",
  "instance_report_csv": "C:\\Users\\edual\\projeto-placentas\\codespace\\v2_rfdetr_seg_large_opt_v1\\artifacts\\reports\\seg_large_r672_auto\\placenta_instance_report_rfdetr.csv",
  "totals_report_csv": "C:\\Users\\edual\\projeto-placentas\\codespace\\v2_rfdetr_seg_large_opt_v1\\artifacts\\reports\\seg_large_r672_auto\\placenta_totals_report_rf